In [211]:
import json
import logging
import boto3


from botocore.exceptions import ClientError

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def generate_text_embeddings(model_id, body, region_name):
    """
    Generate text embedding by using the Cohere Embed model.
    Args:
        model_id (str): The model ID to use.
        body (str) : The reqest body to use.
        region_name (str): The AWS region to invoke the model on
    Returns:
        dict: The response from the model.
    """

    logger.info("Generating text embeddings with the Cohere Embed model %s", model_id)

    accept = '*/*'
    content_type = 'application/json'

    bedrock = boto3.client(service_name='bedrock-runtime', region_name=region_name)

    response = bedrock.invoke_model(
        body=body,
        modelId=model_id,
        accept=accept,
        contentType=content_type,
    )

    logger.info("Successfully generated embeddings with Cohere model %s", model_id)

    return response


In [212]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range, MatchAny

client = QdrantClient(host="3.78.186.81", port=6333)

INFO:httpx:HTTP Request: GET http://3.78.186.81:6333 "HTTP/1.1 200 OK"


In [263]:
region_name = 'eu-central-1'
model_id = 'arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.cohere.embed-v4:0'
input_type = "search_document"
embedding_types = ["float"]

user_query = "Based on the given sources for the Danish legislation, could you please summarize the most important changes regarding eletronic packages and package disposal?"

question = json.dumps({
            "texts": [
                user_query,
                ],
            "input_type": input_type,
            "embedding_types": embedding_types,
        })


response = generate_text_embeddings(model_id=model_id, body=question, region_name=region_name)
response_body = json.loads(response.get('body').read())


INFO:__main__:Generating text embeddings with the Cohere Embed model arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.cohere.embed-v4:0
INFO:__main__:Successfully generated embeddings with Cohere model arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.cohere.embed-v4:0


In [264]:
def retrieve_macro_chunks(query_vector, filters, top_k=5):
    results = client.query_points(
        collection_name="1cc_legislation",
        query=query_vector,
        query_filter=Filter(
            must=[
                FieldCondition(key="chunk_level", match=MatchValue(value="macro")),
                *filters,
            ]
        ),
        limit=top_k,
    ).points

    print(results)

    return results


In [265]:
def retrieve_micro_chunks(query_vector, macro_results, filters, per_macro_k=3):
    micro_results = []
    for macro_point in macro_results:
        children = macro_point.payload.get("children", [])
        if not children:
            continue

        micro_results.extend(
            client.query_points(
                collection_name="1cc_legislation",
                query=query_vector,
                query_filter=Filter(
                    must=[
                        FieldCondition(key="chunk_id", match=MatchAny(any=children)),
                        *filters
                    ]
                ),
                limit=per_macro_k,
            ).points
        )

        print(micro_results)

        
    return micro_results


In [266]:
def rerank_chunks(query_text, retrieved_chunks):
    # retrieved_chunks: [(chunk_text, score, metadata), ...]
    # call cross-encoder model for pairwise relevance
    pairs = [(query_text, c["text"]) for c in retrieved_chunks]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(
        zip(retrieved_chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return [r[0] for r in ranked]


In [267]:
def expand_context(points):
    expanded = []
    seen = set()
    for p in points:
        expanded.append(p)
        prev_id = p.payload.get("prev_chunk_id")
        next_id = p.payload.get("next_chunk_id")

        for neighbor_id in [prev_id, next_id]:
            if neighbor_id and neighbor_id not in seen:
                neighbor = client.retrieve(
                    collection_name="1cc_legislation",
                    ids=[neighbor_id],
                    with_payload=True
                )
                if neighbor:
                    expanded.extend(neighbor)
                    seen.add(neighbor_id)
    return expanded


In [268]:
from datetime import datetime, timezone
from qdrant_client.http import models

def retrieve_legislation(query_embedding, country="denmark", year=2025):
    gte_timestamp = int(datetime(year, 1, 1, tzinfo=timezone.utc).timestamp())
    lte_timestamp = int(datetime(year, 12, 31, tzinfo=timezone.utc).timestamp())

    filters = [
        # FieldCondition(key="country", match=MatchValue(value=country)),
        # FieldCondition(key="publication_date", range=Range(gte=gte_timestamp, lte=lte_timestamp)),
    ]

    # Stage 1: macro retrieval
    macro_points = retrieve_macro_chunks(query_embedding, filters, top_k=5)

    # Stage 2: micro retrieval from selected macros
    micro_points = retrieve_micro_chunks(query_embedding, macro_points, filters, per_macro_k=3)

    # Merge + expand context
    all_points = macro_points + micro_points
    expanded_points = expand_context(all_points)

    # Sort by score descending
    sorted_points = sorted(expanded_points, key=lambda p: p.score, reverse=True)

    # Return as text snippets
    return [
        {
            "text": p.payload["text"],
            "title": p.payload.get("title"),
            "document_id": p.payload.get("document_id"),
            "publication_date": p.payload.get("publication_date"),
            "chunk_level": p.payload.get("chunk_level"),
            "score": p.score,
        }
        for p in sorted_points[:10]
    ]


In [269]:
query = response_body.get("embeddings")
query = query['float'][0]


In [270]:
results = retrieve_legislation(query_embedding=query)

INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"


[ScoredPoint(id='ab3ab7e3-c516-54a1-aa84-0599367be832', version=67, score=0.48551553, payload={'internal_id': 'leg-250498-macro-macro-1', 'document_id': 250498, 'title': 'Lov om ændring af lov om miljøbeskyttelse (Præcisering af anvendelsesområde for udvidet producentansvar for elektrisk og elektronisk udstyr)', 'source': '1cc_legislation', 'publication_date': 1750464000, 'chunk_id': 'macro-1', 'chunk_level': 'macro', 'semantic_density': 1.0, 'keywords': ['stk', 'efter', '2024', 'som', 'indsættes', 'marts', 'elektrisk', 'til'], 'children': [2, 3], 'total_micro_chunks': 7, 'model_id': 'arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.cohere.embed-v4:0', 'text': 'senest ved § 35 i lov nr. 468 af 14. maj 2025, foretages følgende ændringer:\n1. I fodnoten til lovens titel ændres »og dele af« til: »dele af«, og efter »EU-Tidende 2019, nr. L 155, side \n1« indsættes: », og Europa-Parlamentets og Rådets direktiv (EU) 2024/884 af 13. marts 2024 om ændring \naf direktiv 2012/19/EU

INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"


[ScoredPoint(id='ce7631b8-45bc-5228-a89e-001790f72cf7', version=67, score=0.46452606, payload={'internal_id': 'leg-250498-micro-2', 'document_id': 250498, 'title': 'Lov om ændring af lov om miljøbeskyttelse (Præcisering af anvendelsesområde for udvidet producentansvar for elektrisk og elektronisk udstyr)', 'source': '1cc_legislation', 'publication_date': 1750464000, 'section': 'Introduction', 'chunk_id': 2, 'chunk_level': 'micro', 'semantic_density': 1.0, 'keywords': ['2024', 'til', 'dele', 'efter', 'tidende', 'direktiv'], 'prev_chunk_id': 1, 'next_chunk_id': 3, 'parent_macro': 'macro-1', 'total_micro_chunks': 7, 'model_id': 'arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.cohere.embed-v4:0', 'text': 'senest ved § 35 i lov nr. 468 af 14. maj 2025, foretages følgende ændringer:\n1. I fodnoten til lovens titel ændres »og dele af« til: »dele af«, og efter »EU-Tidende 2019, nr. L 155, side \n1« indsættes: », og Europa-Parlamentets og Rådets direktiv (EU) 2024/884 af 13. mart

INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://

In [271]:
results

[{'text': 'Bekendtgørelse om visse krav til emballager, udvidet producentansvar for \nemballage samt øvrigt affald der indsamles med emballageaffald1)\nI medfør af § 7 a, stk. 1, § 9 p, stk. 2, 4, 6-8, 10-14, 16, 18 og 20, § 9 z, stk. 2, 3 og 5-8, § 9 æ, stk. \n1, 2, 4 og 5, § 9 ø, stk. 1, nr. 1-8, og stk. 4, § 9 å, stk. 2 og 3, § 44, stk. 1, § 45, stk. 10 og 11, § 48 d, \nstk. 2, § 51, stk. 1, 5 og 6, § 51 b, § 67, § 79 b, stk. 1-3 og 5, § 79 e, § 80, stk. 1 og 2, og § 110, stk. 3, i',
  'title': 'Bekendtgørelse om visse krav til emballager, udvidet producentansvar for emballage samt øvrigt affald der indsamles med emballageaffald',
  'document_id': 251445,
  'publication_date': 1759190400,
  'chunk_level': 'micro',
  'score': 0.48956856},
 {'text': 'senest ved § 35 i lov nr. 468 af 14. maj 2025, foretages følgende ændringer:\n1. I fodnoten til lovens titel ændres »og dele af« til: »dele af«, og efter »EU-Tidende 2019, nr. L 155, side \n1« indsættes: », og Europa-Parlamentets og Rådet

In [247]:
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0
"""
Shows how to create a list of action items from a meeting transcript
with the Amazon Titan Text model (on demand).
"""
import json
import logging
import boto3

from botocore.exceptions import ClientError


class ImageError(Exception):
    "Custom exception for errors returned by Amazon Titan Text models"

    def __init__(self, message):
        self.message = message


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def generate_text(model_id, body):
    """
    Generate text using Amazon Titan Text models on demand.
    Args:
        model_id (str): The model ID to use.
        body (str) : The request body to use.
    Returns:
        response (json): The response from the model.
    """

    logger.info(
        "Generating text with Amazon Titan Text model %s", model_id)

    bedrock = boto3.client(service_name='bedrock-runtime')

    accept = "application/json"
    content_type = "application/json"

    response = bedrock.invoke_model(
        body=body, modelId=model_id, accept=accept, contentType=content_type,
    )
    response_body = json.loads(response.get("body").read())

    finish_reason = response_body.get("error")

    if finish_reason is not None:
        raise ImageError(f"Text generation error. Error is {finish_reason}")

    logger.info(
        "Successfully generated text with Amazon Titan Text model %s", model_id)

    return response_body



In [248]:
model_id = 'arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.anthropic.claude-3-5-sonnet-20240620-v1:0'

prompt = f""" 

You are Liggy a senior legal consultant specialized in eletronics and batteries. You work a prestigious consulting 
company in europe and your client are other consulting companies responsable to organize and plan eletronics live cycle in Europe.
In order to provide always the most precise and optimal answer to your clients here are some guide lines:
 - Your answer must be always as precise and objective.
 - Yours answers should be as complete as possible, explaining in details the cliens doubts, do not presupose any prior knowledge.
 - As you field act is legal consultancy misintrepretation can lead to desasters, therefore all your answers must be based exclusively on the legislation provided as sources.
 - When using the legal sources, provide details as paragraph, act and other relevant information so the client can also double check the answers.
 - If you think that there is not enought information in the sources provided, you should complete with your own knowledge, however, make it clear for the client.

 LEGAL SOURCES:

 {search_result}

here is the user question you should answer: {user_query}
"""

body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 10000,
    "temperature": 0.7,
        "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        },
    ],
})

response_rag_body = generate_text(model_id, body)

INFO:__main__:Generating text with Amazon Titan Text model arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.anthropic.claude-3-5-sonnet-20240620-v1:0
INFO:__main__:Successfully generated text with Amazon Titan Text model arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.anthropic.claude-3-5-sonnet-20240620-v1:0


In [249]:
response_rag_body

{'id': 'msg_bdrk_01UoxL77JegNrTBYzAh9GejB',
 'type': 'message',
 'role': 'assistant',
 'model': 'claude-3-5-sonnet-20240620',
 'content': [{'type': 'text',
   'text': 'Certainly. Based on the provided legal sources, I can offer some details about the Danish Environmental Protection Act (Lov om miljøbeskyttelse) specifically regarding electronic waste. Here\'s what we can gather from the information available:\n\n1. Recent Amendment:\nThe Environmental Protection Act has been recently amended to clarify the scope of extended producer responsibility for electrical and electronic equipment. This amendment is implemented through the "Lov om ændring af lov om miljøbeskyttelse" (Law amending the Environmental Protection Act).\n\n2. EU Directive Implementation:\nThe amendment incorporates the European Parliament and Council Directive (EU) 2024/884 of March 13, 2024, which modifies Directive 2012/19/EU on waste electrical and electronic equipment (WEEE). This is reflected in the footnote to th

In [ ]:
from datetime import datetime, timezone

query = response_body.get("embeddings")
query = query['float'][0] 
# Convert date strings to UNIX timestamps 
gte_timestamp = int(datetime(2025, 1, 1, tzinfo=timezone.utc).timestamp()) 
lte_timestamp = int(datetime(2025, 12, 31, tzinfo=timezone.utc).timestamp()) 

search_result = client.query_points( collection_name="1cc_legislation",
                                     query=query,
                                              limit=10, ).points

print(search_result)

INFO:httpx:HTTP Request: POST http://3.78.186.81:6333/collections/1cc_legislation/points/query "HTTP/1.1 200 OK"


[ScoredPoint(id='68ea7054-5f5a-5b5e-9a74-35884134083d', version=5, score=0.48956856, payload={'internal_id': 'leg-251445-micro-1', 'document_id': 251445, 'title': 'Bekendtgørelse om visse krav til emballager, udvidet producentansvar for emballage samt øvrigt affald der indsamles med emballageaffald', 'source': '1cc_legislation', 'publication_date': 1759190400, 'section': 'Introduction', 'chunk_id': 1, 'chunk_level': 'micro', 'semantic_density': 0.982, 'keywords': ['stk', 'bekendtgørelse', 'visse', 'krav', 'til', 'emballager'], 'prev_chunk_id': 0, 'next_chunk_id': 2, 'parent_macro': 'macro-0', 'total_micro_chunks': 447, 'model_id': 'arn:aws:bedrock:eu-central-1:407179558514:inference-profile/eu.cohere.embed-v4:0', 'text': 'Bekendtgørelse om visse krav til emballager, udvidet producentansvar for \nemballage samt øvrigt affald der indsamles med emballageaffald1)\nI medfør af § 7 a, stk. 1, § 9 p, stk. 2, 4, 6-8, 10-14, 16, 18 og 20, § 9 z, stk. 2, 3 og 5-8, § 9 æ, stk. \n1, 2, 4 og 5, § 9